# Entrenamiento de Atlas.onnx para openWakeWord

Ruta elegida: Google Colab. El entrenamiento automatico oficial de openWakeWord usa Piper para generar ejemplos sinteticos y esta soportado en Linux; no se asume que el equipo Windows local pueda ejecutarlo de forma fiable.

Este notebook no contiene secretos, no usa Picovoice y no demuestra fiabilidad fisica. La fiabilidad solo se valida despues con el microfono real en Atlas.

In [ ]:
import os
import random
import sys
from pathlib import Path

import numpy as np

SEED = 4310
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
print("Seed:", SEED)


## 1. Instalacion compatible

Ejecuta en Colab con runtime Linux. GPU T4 o superior es recomendable. El repositorio oficial se instala editable para usar `openwakeword/openwakeword/train.py` y `examples/custom_model.yml`.

In [ ]:
!rm -rf openwakeword piper-sample-generator
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -q -O piper-sample-generator/models/en_US-libritts_r-medium.pt https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt
!pip install -q piper-phonemize webrtcvad
!git clone https://github.com/dscripka/openWakeWord openwakeword
!pip install -q -e ./openwakeword
!pip install -q mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.11.0 acoustics==0.2.6 tensorflow-cpu==2.8.1 tensorflow_probability==0.16.0 onnx_tf==1.10.0 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 onnxruntime


In [ ]:
resource_dir = Path("openwakeword/openwakeword/resources/models")
resource_dir.mkdir(parents=True, exist_ok=True)
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget -q https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O openwakeword/openwakeword/resources/models/melspectrogram.tflite
print(sorted(p.name for p in resource_dir.glob('*')))


## 2. Datos positivos, negativos y ruido

El script oficial generara ejemplos positivos de la frase objetivo y frases adversarias. Para negativos/ruido se usan RIRs, Audioset/FMA y features precomputadas de openWakeWord. Aumenta las horas y muestras si el modelo necesita menos falsos positivos.

In [ ]:
import datasets
import scipy.io.wavfile
from tqdm import tqdm

Path("mit_rirs").mkdir(exist_ok=True)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="RIR"):
    name = Path(row["audio"]["path"]).name
    scipy.io.wavfile.write(Path("mit_rirs") / name, 16000, (row["audio"]["array"] * 32767).astype(np.int16))

Path("audioset").mkdir(exist_ok=True)
Path("audioset_16k").mkdir(exist_ok=True)
fname = "bal_train09.tar"
!wget -q -O audioset/{fname} https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/{fname}
!cd audioset && tar -xf {fname}
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(p) for p in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="Audioset 16k"):
    name = Path(row["audio"]["path"]).with_suffix(".wav").name
    scipy.io.wavfile.write(Path("audioset_16k") / name, 16000, (row["audio"]["array"] * 32767).astype(np.int16))


In [ ]:
Path("fma").mkdir(exist_ok=True)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))
n_hours = 1
for index in tqdm(range(n_hours * 3600 // 30), desc="FMA 16k"):
    row = next(fma_dataset)
    name = Path(row["audio"]["path"]).with_suffix(".wav").name
    scipy.io.wavfile.write(Path("fma") / name, 16000, (row["audio"]["array"] * 32767).astype(np.int16))

!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget -q https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy


## 3. Configuracion de entrenamiento

La frase objetivo es `atlas`. El nombre del modelo exportado se fija como `Atlas.onnx` para coincidir con la configuracion local del proyecto.

In [ ]:
import yaml

config = yaml.load(Path("openwakeword/examples/custom_model.yml").read_text(), yaml.Loader)
config["target_phrase"] = ["atlas"]
config["model_name"] = "Atlas"
config["n_samples"] = 5000
config["n_samples_val"] = 1000
config["steps"] = 10000
config["target_accuracy"] = 0.7
config["target_recall"] = 0.5
config["background_paths"] = ["./audioset_16k", "./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

Path("atlas_model.yaml").write_text(yaml.dump(config), encoding="utf-8")
print(Path("atlas_model.yaml").read_text())


## 4. Generacion, augmentacion y entrenamiento

Estas celdas pueden tardar. Si una generacion parcial falla, vuelve a ejecutar la misma celda; el script completa los objetivos del YAML.

In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py --training_config atlas_model.yaml --generate_clips


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py --training_config atlas_model.yaml --augment_clips


In [ ]:
!{sys.executable} openwakeword/openwakeword/train.py --training_config atlas_model.yaml --train_model


## 5. Exportacion, validacion y descarga

La validacion carga `Atlas.onnx` con `openwakeword.model.Model` y ejecuta prediccion sobre PCM int16 silencioso. Esto comprueba compatibilidad tecnica, no calidad acustica.

In [ ]:
from openwakeword.model import Model

model_path = Path("my_custom_model/Atlas.onnx")
if not model_path.is_file():
    raise FileNotFoundError(model_path)

model = Model(wakeword_models=[str(model_path)], inference_framework="onnx")
scores = model.predict(np.zeros(1280, dtype=np.int16))
print(scores)
if not isinstance(scores, dict) or "Atlas" not in scores:
    raise RuntimeError("El modelo no devuelve score Atlas")


In [ ]:
from google.colab import files

files.download(str(model_path))
